# Sistemas de Recomendación

In [40]:
import pandas as pd
import numpy as np

## Similitud coseno

$$sim(\pmb x, \pmb y) = \frac {\pmb x \cdot \pmb y}{||\pmb x|| \cdot ||\pmb y||}$$

¿Cómo calcularla en Python?

Supongamos que tenemos la siguiente matriz:

|  	| Libro A 	| Libro B 	| Libro C 	|
|-------	|---------	|---------	|---------	|
| Juan 	| 5 	| 4 	| 4 	|
| Diego 	| 4 	| 5 	| 5 	|


Podemos calcular la similitud coseno empleando sklearn:

In [41]:
from sklearn.metrics.pairwise import cosine_similarity
Juan = [5,4,4]
Diego = [4,5,5]
cosine_similarity([Juan, Diego])

array([[1.        , 0.97823198],
       [0.97823198, 1.        ]])

También podemos calcular la similitud a mano:

In [3]:
(5*4 + 4*5 + 4*5)/(np.sqrt(5**2+4**2+4**2)*np.sqrt(4**2+5**2+5**2))

0.9782319760890369

O empleando Numpy

Calcular la similitud coseno usando numpy (con np.dot y np.linalg.norm)

In [42]:
np.dot(Juan,Diego)/np.dot(np.linalg.norm(Juan), np.linalg.norm(Diego))

0.9782319760890369

Ahora bien, cuando tenemos una matriz user-item de la vida real, tenemos muchos casos faltantes. En esta situación, no podremos calcular la similitud coseno tan fácilmente...

In [43]:
user_item = np.array([[5, np.nan, 4],[4,3,5],[4,5,5],[np.nan, 5, np.nan], [np.nan, 5, 3]])
user_item

array([[ 5., nan,  4.],
       [ 4.,  3.,  5.],
       [ 4.,  5.,  5.],
       [nan,  5., nan],
       [nan,  5.,  3.]])

## Surprise

En esta notebook vamos a emplear la librería surprise. Esta es una librería que se basa en la API de scikit-learn y permite implementar varios algoritmos básicos de recomendación.

Comencemos cargando un dataset clásico en sistemas de recomendación: MovieLens (https://movielens.org/). Esta es una página de recomendación de películas que abrió información histórica.

In [44]:
!pip list

Package                            Version
---------------------------------- -------------------
absl-py                            1.4.0
accelerate                         1.1.1
aiohappyeyeballs                   2.4.4
aiohttp                            3.11.9
aiosignal                          1.3.1
alabaster                          1.0.0
albucore                           0.0.19
albumentations                     1.4.20
altair                             4.2.2
annotated-types                    0.7.0
anyio                              3.7.1
argon2-cffi                        23.1.0
argon2-cffi-bindings               21.2.0
array_record                       0.5.1
arviz                              0.20.0
astropy                            6.1.7
astropy-iers-data                  0.2024.12.2.0.35.34
astunparse                         1.6.3
async-timeout                      4.0.3
atpublic                           4.1.0
attrs                              24.2.0
audioread           

In [6]:
!pip install surprise
# https://surprise.readthedocs.io/

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.4/154.4 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-surprise: filename=scikit_surprise-1.1.4-cp310-cp310-linux_x86_64.whl size=2357273 sha256=92089159ef7d81eb56c12f6b0c5b12cad27fa1175078ebc3e138d4979c7ac006
  Stored in directory: /root/.cache/pip/wheels/4b/3f/df/6acbf0a40397d9bf3ff97f582cc22fb9ce66adde75bc71fd54
Successfully built scikit-surprise


In [ ]:
# Bajamos el dataset. En windows pueden descargarlo entrando al link manualmente
#!wget https://files.grouplens.org/datasets/movielens/ml-100k/u.data

In [51]:
# información sobre Movie lens https://files.grouplens.org/datasets/movielens/ml-100k-README.txt

mlens = pd.read_csv("https://files.grouplens.org/datasets/movielens/ml-100k/u.data",sep="\t",header=None)
mlens.columns = ["user_id","item_id","rating","timestamp"]
mlens

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
...,...,...,...,...
99995,880,476,3,880175444
99996,716,204,5,879795543
99997,276,1090,1,874795795
99998,13,225,2,882399156


In [47]:
len(mlens['user_id'].unique())

943

In [48]:
len(mlens['item_id'].unique())

1682

In [52]:
mlens.drop("timestamp", axis=1, inplace=True)

In [53]:
mlens

,user_id,item_id,rating
0,196,242,3
1,186,302,3
2,22,377,1
3,244,51,2
4,166,346,1
...,...,...,...
99995,880,476,3
99996,716,204,5
99997,276,1090,1
99998,13,225,2


El paquete surprise no recibe directamente un objeto DataFrame.
Para parsear y leer un conjunto de datos debe hacerlo a través de dos nuevos objetos: Reader y Dataset.
En Reader debemos especificar el valor mínimo y el valor máximo de los ratings.
Dataset nos permite leer datos desde distintas fuentes.

In [54]:
from surprise import Dataset, Reader
reader = Reader(rating_scale=(mlens["rating"].min(),mlens["rating"].max()))

In [55]:
dataset = Dataset.load_from_df(mlens,reader)
dataset

Ahora cargue SVD y GridSearchCV, ambos de surprise.

In [56]:
from surprise import SVD
from surprise.model_selection.search import GridSearchCV
'''
classsurprise.prediction_algorithms.matrix_factorization.SVD(n_factors=100,
n_epochs=20, biased=True, init_mean=0, init_std_dev=0.1, lr_all=0.005, reg_all=0.02,
lr_bu=None, lr_bi=None, lr_pu=None, lr_qi=None, reg_bu=None, reg_bi=None,
reg_pu=None, reg_qi=None, random_state=None, verbose=False)

class surprise.model_selection.search.GridSearchCV(algo_class, param_grid,
measures=['rmse', 'mae'], cv=None, refit=False, return_train_measures=False,
n_jobs=1, pre_dispatch='2*n_jobs', joblib_verbose=0)
'''


"\nclasssurprise.prediction_algorithms.matrix_factorization.SVD(n_factors=100,\nn_epochs=20, biased=True, init_mean=0, init_std_dev=0.1, lr_all=0.005, reg_all=0.02,\nlr_bu=None, lr_bi=None, lr_pu=None, lr_qi=None, reg_bu=None, reg_bi=None,\nreg_pu=None, reg_qi=None, random_state=None, verbose=False)\n\nclass surprise.model_selection.search.GridSearchCV(algo_class, param_grid,\nmeasures=['rmse', 'mae'], cv=None, refit=False, return_train_measures=False,\nn_jobs=1, pre_dispatch='2*n_jobs', joblib_verbose=0)\n"

Genere una grilla de parámetros donde se prueben distintas combinaciones de:  
  - epochs: es la cantidad de pasadas sobre el dataset que hará el algoritmo empleando descenso por el gradiente  
  - biased: usar parámetros de sesgo o no  
  - lr_all: learning rate para todos los parámetros  
  - reg_all: término de regularización para todos los parámetros (lambda)  

In [57]:
param_grid = {'n_epochs': [5, 10], 'lr_all': [0.002, 0.005], 'reg_all': [0.4, 0.6]}

Emplee GridSearchCV, SVD y el diccionario con los parámetros para probar, y entrene un modelo. Note que a GridSearchCV necesita pasarle un modelo sin instanciar. Además, setee el parámetro refit a True y con measures = ["rmse","fcp"]

In [58]:
gs = GridSearchCV(SVD, param_grid, measures=['fcp',"rmse"], cv=3, refit=True)

In [59]:
gs.fit(dataset)

Imprima el rmse y el fcp, y la mejor combinación de parámetros

In [60]:
gs.best_score

{'fcp': 0.6985552121591029, 'rmse': 0.963533832255186}

In [61]:
gs.best_params

{'fcp': {'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.4},
 'rmse': {'n_epochs': 10, 'lr_all': 0.005, 'reg_all': 0.4}}

Guarde el modelo con mayor fcp y prediga el rating para el user id 196 e item id 242

In [62]:
best_model = gs.best_estimator["fcp"]

In [19]:
pred = best_model.predict("196", "242")
pred

Prediction(uid='196', iid='242', r_ui=None, est=3.52986, details={'was_impossible': False})

In [20]:
pred.est

3.52986

Pruebe empleando otros modelos como SVDpp,  NMF,  KNNWithZScore e intente superar el valor obtenido

In [63]:
from surprise import SVDpp

'''
class surprise.prediction_algorithms.matrix_factorization.SVDpp(n_factors=20, n_epochs=20,
init_mean=0, init_std_dev=0.1, lr_all=0.007, reg_all=0.02, lr_bu=None, lr_bi=None,
lr_pu=None, lr_qi=None, lr_yj=None, reg_bu=None, reg_bi=None, reg_pu=None, reg_qi=None,
reg_yj=None, random_state=None, verbose=False, cache_ratings=False)
'''

gs1 = GridSearchCV(SVDpp, param_grid, measures=['fcp',"rmse"], cv=3, refit=True)
gs1.fit(dataset)
gs1.best_score

{'fcp': 0.6972050862116247, 'rmse': 0.9646864118401878}

In [22]:
from surprise import NMF

'''
class surprise.prediction_algorithms.matrix_factorization.NMF(n_factors=15, n_epochs=50,
biased=False, reg_pu=0.06, reg_qi=0.06, reg_bu=0.02, reg_bi=0.02, lr_bu=0.005,
lr_bi=0.005, init_low=0, init_high=1, random_state=None, verbose=False)
'''

param_grid2 = {'n_epochs': [5, 10], 'lr_bu': [0.002, 0.005], 'lr_bi': [0.002, 0.005],
               'reg_pu': [0.04, 0.06],'reg_qi': [0.04, 0.06],'reg_bu': [0.01, 0.03],'reg_bi': [0.01, 0.03]}

gs2 = GridSearchCV(NMF, param_grid2, measures=['fcp',"rmse"], cv=3, refit=True)
gs2.fit(dataset)
gs2.best_score

{'fcp': 0.6818098677673348, 'rmse': 0.997309772843237}

In [23]:
from surprise import KNNWithZScore

'''
classsurprise.prediction_algorithms.knns.KNNWithZScore(k=40, min_k=1,
sim_options={}, verbose=True, **kwargs)
'''
param_grid3={'k': range(20, 65,5)}
gs3 = GridSearchCV(KNNWithZScore, param_grid3, measures=['fcp',"rmse"], cv=3, refit=True)
gs3.fit(dataset)
gs3.best_score

Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computing similarity matrix.
Computing the msd similarity matrix...
Done computi

{'fcp': 0.7060784337366138, 'rmse': 0.9558445998537705}

# Recomendación basada en el contenido

En este ejemplo vamos a tomar un corpus de textos de autores latinoamericanos para sugerir uno similar a uno dado. Para esto construiremos una matriz TFIDF, de frecuencias normalizadas de términos por documento, y usaremos la similitud coseno para medir distancias entre los distintos textos.

In [64]:
!git clone https://github.com/karen-pal/borges

fatal: destination path 'borges' already exists and is not an empty directory.


In [65]:
!ls


borges	sample_data


In [67]:
!cd borges
!ls borges/

borges.ipynb  full_text_scrapper.py  link_scrapper.py  scraper.py		  uso_simple.ipynb
datasets      LDA.ipynb		     README.md	       sentence_similarity.ipynb


In [68]:
!ls borges/datasets

borges_sentiment_corpus.csv  datasets_csv  datasets_pkl  full_corpus.csv  links  pkl_to_csv.ipynb


In [69]:
import pickle
from pathlib import Path
import pandas as pd

df = pd.DataFrame()
# usando el asterisco de "wildcard" traemos todos los archivos en formato pickle
pkls = Path('.').glob('./borges/datasets/datasets_pkl/*texts.pkl')

# leemos todos los pickles y concatenarlos en un DataFrame
for pkl in pkls:
    with open(pkl, 'rb') as inp:
        df_ = pickle.load(inp)
    df = pd.concat([df, df_])

df.shape

(719, 3)

In [70]:
df

,link,text_metadata,text
0,https://ciudadseva.com/texto/el-alma/,"{'title': 'El alma', 'metadata': '[Cuento - Te...",I ¿Qué viene a buscar el Diablo en mi aposento...
1,https://ciudadseva.com/texto/el-difunto-y-yo/,"{'title': 'El difunto y yo', 'metadata': '[Cue...",Examiné apresuradamente la extraña situación e...
2,https://ciudadseva.com/texto/el-pequeno-nazareno/,"{'title': 'El pequeño nazareno', 'metadata': '...","El miércoles santo, el pequeño Nazareno de tún..."
3,https://ciudadseva.com/texto/la-tienda-de-mune...,"{'title': 'La tienda de muñecos', 'metadata': ...","No sé cuándo, dónde ni por quién fue escrito e..."
0,https://ciudadseva.com/texto/el-crepusculo-del...,"{'title': 'El crepúsculo del diablo', 'metadat...",En el borde de una pila que muestra su cuenca ...
...,...,...,...
20,https://ciudadseva.com/texto/tempestad-de-almas/,"{'title': 'Tempestad de almas', 'metadata': '[...","Ah, si lo hubiera sabido, no nacía, ah, si lo ..."
21,https://ciudadseva.com/texto/un-caso-complicado/,"{'title': 'Un caso complicado', 'metadata': '[...","Pues sí. Cuyo padre era amante, con un alfiler..."
22,https://ciudadseva.com/texto/una-gallina/,"{'title': 'Una gallina', 'metadata': '[Cuento ...",Era una gallina de domingo. Todavía vivía porq...
23,https://ciudadseva.com/texto/una-tarde-plena/,"{'title': 'Una tarde plena', 'metadata': '[Cue...","El saguino¹ es tan pequeño como un ratón, y de..."


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


No charts were generated by quickchart


In [72]:
df['text_metadata'].sample(2)

,text_metadata
19,"{'title': 'La mano pegada', 'metadata': '[Cuen..."
6,"{'title': 'El rey burgués', 'metadata': '[Cuen..."


In [73]:
df.columns

Index(['link', 'text_metadata', 'text'], dtype='object')

In [74]:
# separamos de la metadata el título y autor en sus propias columnas
df['title'] = df['text_metadata'].apply(lambda x: x['title'])
df['author'] = df['text_metadata'].apply(lambda x: x['author'])

In [75]:
# vemos los autores disponibloes
df['author'].value_counts()

,count
author,
Jorge Luis Borges,60
Julio Cortázar,55
Baldomero Lillo,50
Augusto Monterroso,45
Juan José Arreola,45
Alfonso Reyes,37
Enrique Anderson Imbert,36
Mario Benedetti,33
Julio Ramón Ribeyro,27


In [76]:
# quitamos duplicados y reiniciamos el índice
df = df.drop_duplicates(subset=[c for c in df.columns if c != 'text_metadata'])
df = df.reset_index(drop=True)
df.shape

(693, 5)

In [77]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import linear_kernel
from pprint import pprint

Vamos a calcular las matrices de ocurrencias de términos usando sklearn.

Ámbas clases primero construyen el vocabulario total, y luego:  
- **CountVectorizer** nos devuelve la frecuencia absoluta de cada término por cada documento.
- [**TF-IDF**](https://en.wikipedia.org/wiki/Tf%E2%80%93idf): calcula la frecuencia de cada término por documento, y normaliza por el total de documentos donde el término aparece.

$${tf} (t,d)={\frac {f_{t,d}}{\sum _{t'\in d}{f_{t',d}}}}$$

$$
idf( t, D ) = log \frac{ \text{| } D \text{ |} }{ 1 + \text{| } \{ d \in D : t \in d \} \text{ |} }
$$


$$ tfidf( t, d, D ) = tf( t, d ) \times idf( t, D )
$$


In [78]:
# Instanciamos el CV
vectorizer = CountVectorizer()

doc1 = 'la matriz de frecuencias por palabras otorga información del contenido de un documento'
doc2 = 'las palabras que aparecen en un documento se relaciona con su tema'
# Definimos una lista con todos los strings
data_corpus = [doc1, doc2]

# Fiteamos el CV y transformamos los datos
X = vectorizer.fit_transform(data_corpus)

# Pasamos de sparse matrix a array usando .toarray()

print(X.toarray())
# Usando el metodo .get_feature_names() del CV podemos acceder al indice de palabras
print(vectorizer.get_feature_names_out())

[[0 0 1 2 1 1 0 1 1 1 0 1 1 1 1 0 0 0 0 0 1]
 [1 1 0 0 0 1 1 0 0 0 1 0 0 1 0 1 1 1 1 1 1]]
['aparecen' 'con' 'contenido' 'de' 'del' 'documento' 'en' 'frecuencias'
 'información' 'la' 'las' 'matriz' 'otorga' 'palabras' 'por' 'que'
 'relaciona' 'se' 'su' 'tema' 'un']


In [79]:
X

<2x21 sparse matrix of type '<class 'numpy.int64'>'
	with 24 stored elements in Compressed Sparse Row format>

In [80]:
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

stop = list(stopwords.words('spanish'))
# eliminamos las "stop words", palabras comunes no informativas
tf = TfidfVectorizer(stop_words=stop)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [81]:
# calculamos los features para cada ítem (texto)
tfidf_matrix = tf.fit_transform(df['text'])

In [87]:
tfidf_matrix.shape

(693, 56608)

In [83]:
# calculamos las similitudes entre todos los documentos
cosine_similarities = linear_kernel(tfidf_matrix, tfidf_matrix)
n = 6

# diccionario creado para guardar el resultado en un formato (autor - titulo : puntaje, titulo, autor)
results = {}
for idx, row in df.iterrows():
    # guardamos los indices similares basados en la similitud coseno. Los ordenamos en modo ascendente, siendo 0 nada de similitud y 1 total
    similar_indices = cosine_similarities[idx].argsort()[:-n-2:-1]
    # guardamos los N más cercanos
    similar_items = [(f"{df['author'][i]} - {df['title'][i]}", round(cosine_similarities[idx][i], 3)) for i in similar_indices]
    results[f"{row['author']} - {row['title']}"] = similar_items[1:]

In [84]:
pprint(results['Jorge Luis Borges - El Aleph'])

[('Jorge Luis Borges - La escritura del dios', 0.144),
 ('Jorge Luis Borges - El inmortal', 0.135),
 ('Jorge Luis Borges - Utopía de un hombre que está cansado', 0.125),
 ('Felisberto Hernández - El acomodador', 0.122),
 ('Clarice Lispector - La búsqueda de la dignidad', 0.121),
 ('Jorge Luis Borges - Funes el memorioso', 0.11)]


In [85]:
def recomendar(autor, titulo):
    pprint(results[f"{autor} - {titulo}"])

In [86]:
recomendar('Julio Cortázar', 'Axolotl')

[('Felisberto Hernández - El acomodador', 0.134),
 ('Felisberto Hernández - El cocodrilo', 0.101),
 ('Felisberto Hernández - Menos Julia', 0.089),
 ('Julio Cortázar - Después del almuerzo', 0.088),
 ('Julio Cortázar - La noche boca arriba', 0.086),
 ('Julio Cortázar - La señorita Cora', 0.086)]


In [ ]:
# https://medium.com/@eng.saavedra/sistemas-de-recomendaci%C3%B3n-parte-2-b8a5dc9dc730
# https://towardsdatascience.com/item-based-collaborative-filtering-in-python-91f747200fab
# https://www.genbeta.com/web/asi-funcionan-las-recomendaciones-de-amazon
